# Rebuild MobileCLIP-S0 LiteRT assets

Recreates the three MobileCLIP files used by the backend:

| File | Contract |
| --- | --- |
| `mobileclip_s0_text.tflite` | inputs `int32[1, 77]` token IDs and `int32[1]` end-of-text position → `float32[1, 512]` |
| `mobileclip_s0_image.tflite` | input `float32[1, 3, 256, 256]` RGB in `[0, 1]` → `float32[1, 512]` |
| `mobileclip_s0_tokenizer.json` | Hugging Face `tokenizers` file, vocabulary 49408, SOT 49406, EOT 49407 |

Run in Google Colab (CPU runtime is enough): **Runtime → Run all**. The last cell downloads the files; copy them into the project's `models/` folder and run `python tools/verify_models.py`.

Conversion-only packages (PyTorch, `litert-torch`, the MobileCLIP source) stay in this notebook and are not runtime dependencies.

In [ ]:
%%bash
set -e
pip install -q litert-torch transformers timm open_clip_torch ftfy
if [ ! -d ml-mobileclip ]; then
  git clone -q https://github.com/apple/ml-mobileclip.git
fi
# The repository was restructured for MobileCLIP2 (2025). If the original
# v1 factory is gone, fall back to the last commit before that change.
if ! grep -q "def create_model_and_transforms" ml-mobileclip/mobileclip/__init__.py; then
  COMMIT=$(git -C ml-mobileclip rev-list -n 1 --before=2025-08-01 HEAD)
  echo "Checking out v1 commit $COMMIT"
  git -C ml-mobileclip checkout -q "$COMMIT"
fi
if [ ! -f mobileclip_s0.pt ]; then
  wget -q -O mobileclip_s0.pt \
    https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s0.pt
fi
ls -lh mobileclip_s0.pt

In [ ]:
import hashlib
import sys

import numpy as np
import torch

sys.path.insert(0, "/content/ml-mobileclip")
import mobileclip  # noqa: E402

torch.manual_seed(0)

# Week 3 finding: the factory already returns a reparameterized model, and
# calling reparameterize_model() again raises AttributeError, so it is not
# called here.
model, _, preprocess = mobileclip.create_model_and_transforms(
    "mobileclip_s0", pretrained="mobileclip_s0.pt"
)
model.eval()
reference_tokenizer = mobileclip.get_tokenizer("mobileclip_s0")

assert hasattr(model, "text_encoder") and hasattr(model, "image_encoder")
assert hasattr(model.text_encoder, "projection_layer")
print("Loaded MobileCLIP-S0")

In [ ]:
TEXT_LENGTH = 77
IMAGE_SIZE = 256


class TextEncoderExport(torch.nn.Module):
    """Text tower with the end-of-text position as an explicit input.

    The original model selects the EOT token with argmax inside the graph;
    passing the position in keeps the exported graph simple. The backend
    computes it as argmax(token_ids), which is the same position because
    EOT (49407) is the largest token ID.
    """

    def __init__(self, text_encoder: torch.nn.Module) -> None:
        super().__init__()
        self.text_encoder = text_encoder

    def forward(self, token_ids: torch.Tensor, eot_position: torch.Tensor) -> torch.Tensor:
        hidden = self.text_encoder(token_ids.to(torch.long), return_all_tokens=True)
        selected = torch.index_select(hidden, 1, eot_position.to(torch.long))
        return selected[:, 0, :] @ self.text_encoder.projection_layer


class ImageEncoderExport(torch.nn.Module):
    def __init__(self, clip_model: torch.nn.Module) -> None:
        super().__init__()
        self.clip_model = clip_model

    def forward(self, pixels: torch.Tensor) -> torch.Tensor:
        return self.clip_model.encode_image(pixels)


text_module = TextEncoderExport(model.text_encoder).eval()
image_module = ImageEncoderExport(model).eval()

sample_tokens = reference_tokenizer(["a photo of a cat"]).to(torch.int32)
sample_eot = sample_tokens.argmax(dim=-1).to(torch.int32)
sample_pixels = torch.rand(1, 3, IMAGE_SIZE, IMAGE_SIZE)

with torch.no_grad():
    wrapped = text_module(sample_tokens, sample_eot)
    original = model.encode_text(sample_tokens.to(torch.long))

cosine = torch.nn.functional.cosine_similarity(wrapped, original).item()
print("Wrapper vs encode_text cosine:", cosine)
assert cosine > 0.9999, "Text wrapper does not reproduce encode_text"
assert tuple(sample_tokens.shape) == (1, TEXT_LENGTH)

In [ ]:
import litert_torch

edge_text = litert_torch.convert(text_module, (sample_tokens, sample_eot))
edge_text.export("mobileclip_s0_text.tflite")

edge_image = litert_torch.convert(image_module, (sample_pixels,))
edge_image.export("mobileclip_s0_image.tflite")
print("Exported both encoders")

In [ ]:
from ai_edge_litert.interpreter import Interpreter


def run_tflite(path, inputs_by_shape):
    interpreter = Interpreter(model_path=path)
    interpreter.allocate_tensors()
    details = interpreter.get_input_details()
    for detail in details:
        interpreter.set_tensor(detail["index"], inputs_by_shape[tuple(detail["shape"])])
    interpreter.invoke()
    output = interpreter.get_output_details()
    assert len(output) == 1 and tuple(output[0]["shape"]) == (1, 512), output
    return details, interpreter.get_tensor(output[0]["index"])


def report(name, litert_output, torch_output):
    a = np.asarray(litert_output, dtype=np.float32).ravel()
    b = torch_output.detach().numpy().astype(np.float32).ravel()
    cos = float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
    print(f"{name}: max abs error {np.abs(a - b).max():.3e}, cosine {cos:.6f}")
    assert cos > 0.9999, f"{name} parity check failed"


prompts = ["a diagram", "a dog", "a cat", "Offline Local Retrieval, 2026!"]

for prompt in prompts:
    tokens = reference_tokenizer([prompt]).to(torch.int32)
    eot = tokens.argmax(dim=-1).to(torch.int32)
    details, out = run_tflite(
        "mobileclip_s0_text.tflite",
        {(1, TEXT_LENGTH): tokens.numpy(), (1,): eot.numpy()},
    )
    with torch.no_grad():
        report(f"text '{prompt}'", out, model.encode_text(tokens.to(torch.long)))

input_contract = {(tuple(d["shape"]), np.dtype(d["dtype"]).name) for d in details}
assert input_contract == {((1, TEXT_LENGTH), "int32"), ((1,), "int32")}, details

for seed in range(3):
    pixels = torch.rand(1, 3, IMAGE_SIZE, IMAGE_SIZE, generator=torch.Generator().manual_seed(seed))
    details, out = run_tflite("mobileclip_s0_image.tflite", {(1, 3, IMAGE_SIZE, IMAGE_SIZE): pixels.numpy()})
    with torch.no_grad():
        report(f"image seed {seed}", out, model.encode_image(pixels))

assert np.dtype(details[0]["dtype"]) == np.float32

In [ ]:
from transformers import CLIPTokenizerFast

# MobileCLIP uses the standard CLIP BPE vocabulary. Export it in the
# Hugging Face `tokenizers` format that the backend loads offline.
hf_tokenizer = CLIPTokenizerFast.from_pretrained("openai/clip-vit-base-patch32")
backend_tokenizer = hf_tokenizer.backend_tokenizer
backend_tokenizer.no_padding()
backend_tokenizer.no_truncation()
backend_tokenizer.save("mobileclip_s0_tokenizer.json")

from tokenizers import Tokenizer

exported = Tokenizer.from_file("mobileclip_s0_tokenizer.json")
assert exported.get_vocab_size() == 49408
assert exported.token_to_id("<|startoftext|>") == 49406
assert exported.token_to_id("<|endoftext|>") == 49407

checks = prompts + ["Hello   World", "离线检索", "PDFium extracts text; 3.5 MB file."]
for text in checks:
    ids = exported.encode(text).ids[:TEXT_LENGTH]
    expected = reference_tokenizer([text])[0].tolist()
    padded = ids + [0] * (TEXT_LENGTH - len(ids))
    status = "match" if padded == expected else "DIFFERENT"
    print(f"{status:9} {text!r}")

In [ ]:
from PIL import Image
from pathlib import Path

figure = Path("ml-mobileclip/docs/fig_accuracy_latency.png")

if figure.is_file():
    with Image.open(figure) as source:
        rgb = source.convert("RGB")
    labels = ["a diagram", "a dog", "a cat"]
    image_tensor = preprocess(rgb).unsqueeze(0)
    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        text_features = model.encode_text(reference_tokenizer(labels))
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)[0]
    print({label: round(float(p), 4) for label, p in zip(labels, probs)})
    print("Best label:", labels[int(probs.argmax())])
else:
    print("Reference figure not found; skipping the qualitative check")

In [ ]:
from google.colab import files

for name in [
    "mobileclip_s0_text.tflite",
    "mobileclip_s0_image.tflite",
    "mobileclip_s0_tokenizer.json",
]:
    digest = hashlib.sha256(open(name, "rb").read()).hexdigest()
    size = len(open(name, "rb").read()) / (1024 * 1024)
    print(f"{name:32} {size:7.1f} MB  {digest}")
    files.download(name)